In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import csv
import re

In [2]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://yandex.ru/maps/?display-text=%D0%9F%D0%B0%D0%BD%D1%81%D0%B8%D0%BE%D0%BD%D0%B0%D1%82%20%D0%B4%D0%BB%D1%8F%20%D0%BF%D0%BE%D0%B6%D0%B8%D0%BB%D1%8B%D1%85%20%D0%BB%D1%8E%D0%B4%D0%B5%D0%B9%2C%20%D0%BF%D1%80%D0%B5%D1%81%D1%82%D0%B0%D1%80%D0%B5%D0%BB%D1%8B%D1%85%20%D0%B8%20%D0%B8%D0%BD%D0%B2%D0%B0%D0%BB%D0%B8%D0%B4%D0%BE%D0%B2&ll=37.658051%2C55.598923&mode=search&sctx=ZAAAAAgBEAAaKAoSCZm36jpU2UJAEUIj2Lj%2B2UtAEhIJA15m2CjrAEARz4b8M4N45z8iBgABAgMEBSgKOABAAUgBagJydZ0BzczMPaABAKgBAL0BRUbtU8IBjgHTzfi3%2FwOQzMXi1ALusfDYgAK8uYqM9gP0mpn%2FA8rWtLUFy5rDrbMFp96y3gPXtdKIBvaGhbL9ApnY8YGMA9m59%2BnMAfrq5eMstufemJwFgOW9gPUCp4LszJoDloSX0rEG2LyhhNoDutXj3AbNy6zlBfKCwu2kBdudhsKdBp3Pjf62A5CrlZWqBMOFwfw3ggIbKChjYXRlZ29yeV9pZDooMTg0MTA2MzA4KSkpigIJMTg0MTA2MzA4kgIAmgIMZGVza3RvcC1tYXBzqgLvATIwODYzNDI0MjYzMywyMDY5ODgzODMwNTIsNTI2MzE4NTcwNDMsMTQwMzI3OTAxODMzLDIwODYxOTM3MzgyOCwxMTY5MDExMTM4MTUsODU2ODUyODQ0MzQsMTYzNzI0OTc2MTQ0LDE4ODg1MTAwNzI3NiwxOTAzNTk4NjcyNCwxMjI1NjExMjYyMTYsMTU2NzgyNzkyMTY4LDE2MTQ3NzI2MzU1NiwxNjUxMDMzNzA4MzksMzExNzIxMjEzNjksNjI1MzE2MjQ0MzUsMTYzODYwMTAyNDY4LDEzMTk4OTEyOTA3MCw4ODkxNDgzNjgy2gIoChIJWZpbIazgQkARKwyFVifHS0ASEgkwbXGNz6QJQBFgBmL62trxP%2BACAQ%3D%3D&sll=37.658051%2C55.598923&sspn=2.790527%2C0.970402&text=%7B%22text%22%3A%22%D0%9F%D0%B0%D0%BD%D1%81%D0%B8%D0%BE%D0%BD%D0%B0%D1%82%20%D0%B4%D0%BB%D1%8F%20%D0%BF%D0%BE%D0%B6%D0%B8%D0%BB%D1%8B%D1%85%20%D0%BB%D1%8E%D0%B4%D0%B5%D0%B9%2C%20%D0%BF%D1%80%D0%B5%D1%81%D1%82%D0%B0%D1%80%D0%B5%D0%BB%D1%8B%D1%85%20%D0%B8%20%D0%B8%D0%BD%D0%B2%D0%B0%D0%BB%D0%B8%D0%B4%D0%BE%D0%B2%22%2C%22what%22%3A%5B%7B%22attr_name%22%3A%22category_id%22%2C%22attr_values%22%3A%5B%22184106308%22%5D%7D%5D%7D&z=9"
driver.get(url)
time.sleep(5)

scroll_container = driver.find_element(By.CSS_SELECTOR, "div.scroll__container")

prev_count = 0
no_change_count = 0
for i in range(40):
    driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
    time.sleep(4)
    
    cards = driver.find_elements(By.CSS_SELECTOR, "li.search-snippet-view")
    current_count = len(cards)
    print(f"   Шаг {i+1}: загружено {current_count} организаций")
    
    if current_count == prev_count:
        print("Достигнут конец списка")
        break
    else:
        prev_count = current_count

cards = driver.find_elements(By.CSS_SELECTOR, "li.search-snippet-view")
results = []


for card in cards:
    try:
        #Название+ссылка
        name = ""
        link_href = ""  
        try:
            link = card.find_element(By.CSS_SELECTOR, "a[class*='link-overlay']")
            name = link.get_attribute("aria-label")
            link_href = link.get_attribute("href")  
        except:
            pass
        
        if not name:
            try:
                title = card.find_element(By.CSS_SELECTOR, "div[class*='title']")
                name = title.text.strip()
            except:
                continue
        
        if not name or len(name) < 3:
            continue
        
        # Адрес
        address = "Не указан"
        try:
            addr = card.find_element(By.CSS_SELECTOR, "[class*='address']")
            address = addr.text.strip()
        except:
            pass

        # Рейтинг
        rating = 0.0
        try:
            rating_elem = card.find_element(By.CSS_SELECTOR, "[class*='rating-text']")
            rating_str = rating_elem.text.strip().replace(',', '.')
            rating = float(rating_str)
        except:
            rating = 0.0

        #Число отзывов
        try:
            reviews_elem = card.find_element(By.CSS_SELECTOR, "span.business-rating-amount-view")
            reviews_text = reviews_elem.text.strip()
            reviews_match = re.search(r'(\d+)', reviews_text)
            if reviews_match:
                reviews_count = int(reviews_match.group(1))
            else:
                reviews_count = 0
        except:
            reviews_count = 0
        
        results.append({
            "название": name,
            "адрес": address,
            "рейтинг": rating,
            "число отзывов": reviews_count,
            "ссылка": link_href
        })
        print(f"{name[:50]}")
        
    except Exception as e:
        continue

driver.quit()

name_counts = {}
for r in results:
    name = r['название']
    name_counts[name] = name_counts.get(name, 0) + 1
for r in results:
    i = name_counts[r['название']]
    r['кол-во по сети'] = i if i>0  else "нет"


with open('пансионаты.csv', 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['название', 'адрес', 'рейтинг', 'число отзывов', 'ссылка', 'кол-во по сети'])
    writer.writeheader()
    writer.writerows(results)

print(f"\n ИТОГО: {len(results)} пансионатов")

   Шаг 1: загружено 12 организаций
   Шаг 2: загружено 19 организаций
   Шаг 3: загружено 24 организаций
   Шаг 4: загружено 30 организаций
   Шаг 5: загружено 37 организаций
   Шаг 6: загружено 44 организаций
   Шаг 7: загружено 51 организаций
   Шаг 8: загружено 58 организаций
   Шаг 9: загружено 66 организаций
   Шаг 10: загружено 74 организаций
   Шаг 11: загружено 82 организаций
   Шаг 12: загружено 92 организаций
   Шаг 13: загружено 92 организаций
Достигнут конец списка
Дружный Берег
Пансионат для пожилых людей Вгармонии
Аннушка
Любимые люди
Kartino Village Club
Пансионат для пожилых Бабушки и Дедушки
SM-pension
Теплые беседы
Климовский дом-интернат для престарелых и инвалидо
Океан милосердия
Оазис
Золотая пора
Академия Долголетия
Теплые беседы
Геронтологический центр Юго-западный филиал Геронт
Троицк-1
Сениор Групп
Доброта
Дружный Берег
Радость жизни
Опека
Опека
GreenDay
Пансионат Ховрино
Купава Социальный Гериатрический центр
100+
Vip-пенсионер
Пансионат для пожилых Медик
Панс

In [3]:
with open('пансионаты.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    all_orgs = list(reader)


options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

for i, org in enumerate(all_orgs):
    print(f"\n {i+1}: {org['название']}")
    
    prices_url = org['ссылка'] + "prices/"
    prices = []
    
    try:
        driver.get(prices_url)
        time.sleep(2)
        
        items = driver.find_elements(By.CSS_SELECTOR, "[class*='related-product-view']")
        
        for item in items:
            try:
                title_elem = item.find_element(By.CSS_SELECTOR, "[class*='title']")
                service = title_elem.text.strip()
                
                price_elem = item.find_element(By.CSS_SELECTOR, "[class*='price']")
                price_text = price_elem.text.strip()
                price_match = re.search(r'(\d+(?:\s?\d+)*)', price_text)
                price = price_match.group(1).replace(' ', '') if price_match else ""
                
                prices.append(f"{service}: {price}")
            except:
                continue
        
        org['цены'] = "; ".join(prices) if prices else None
        org['количество_услуг'] = len(prices)
        print(f"Услуг: {len(prices)}")
        
    except Exception as e:
        print(f"Ошибка: {e}")
        org['цены'] = None
        org['количество_услуг'] = 0
    
    time.sleep(1)

driver.quit()

with open('пансионаты.csv', 'w', newline='', encoding='utf-8-sig') as f:
    fieldnames = ['название', 'адрес', 'рейтинг', 'число отзывов', 'ссылка', 'кол-во по сети', 'цены', 'количество_услуг']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_orgs)


total_services = sum(int(o.get('количество_услуг', 0)) for o in all_orgs)
print(f"   Всего услуг/цен собрано: {total_services}")



 1: Дружный Берег
Услуг: 0

 2: Пансионат для пожилых людей Вгармонии
Услуг: 6

 3: Аннушка
Услуг: 0

 4: Любимые люди
Услуг: 0

 5: Kartino Village Club
Услуг: 13

 6: Пансионат для пожилых Бабушки и Дедушки
Услуг: 6

 7: SM-pension
Услуг: 4

 8: Теплые беседы
Услуг: 81

 9: Климовский дом-интернат для престарелых и инвалидов
Услуг: 0

 10: Океан милосердия
Услуг: 0

 11: Оазис
Услуг: 0

 12: Золотая пора
Услуг: 0

 13: Академия Долголетия
Услуг: 12

 14: Теплые беседы
Услуг: 78

 15: Геронтологический центр Юго-западный филиал Геронтологический центр для ветеранов войны Коньково
Услуг: 0

 16: Троицк-1
Услуг: 3

 17: Сениор Групп
Услуг: 3

 18: Доброта
Услуг: 3

 19: Дружный Берег
Услуг: 0

 20: Радость жизни
Услуг: 27

 21: Опека
Услуг: 13

 22: Опека
Услуг: 13

 23: GreenDay
Услуг: 0

 24: Пансионат Ховрино
Услуг: 0

 25: Купава Социальный Гериатрический центр
Услуг: 25

 26: 100+
Услуг: 0

 27: Vip-пенсионер
Услуг: 3

 28: Пансионат для пожилых Медик
Услуг: 0

 29: Пансионат для 

In [4]:
with open('пансионаты.csv', 'r', encoding='utf-8-sig') as f:
    all_orgs = list(csv.DictReader(f))


for org in all_orgs:
    if 'положительных_слов' not in org:
        org['положительных_слов'] = ''
    if 'отрицательных_слов' not in org:
        org['отрицательных_слов'] = ''
    if 'пример_отзыва' not in org:
        org['пример_отзыва'] = ''

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

positive_words = ['хорошо', 'отлично', 'спасибо', 'комфорт', 'чисто', 'внимание', 'забота', 'уют', 'профессионально', 'качественно', 'рекомендую', 'довольны', 'хороший']
negative_words = ['плохо', 'ужасно', 'грязно', 'грубый', 'невнимание', 'дорого', 'ремонт', 'тесно', 'жалко', 'разочарован', 'неудобно', 'проблем']

for i, org in enumerate(all_orgs):
    print(f"\n {i+1}: {org['название']}")
    
    reviews_url = org['ссылка'] + "reviews/"
    reviews = []
    
    try:
        driver.get(reviews_url)
        time.sleep(2)
        
        try:
            reviews_container = driver.find_element(By.CSS_SELECTOR, "div.scroll__container")
            for i in range(2):
                driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", reviews_container)
                time.sleep(1)
        except:
            pass
        
        review_elements = driver.find_elements(By.CSS_SELECTOR, "[class*='spoiler-view__text-container']")
        
        for review in review_elements[:5]:
            text = review.text.strip()
            if text and len(text) > 20:
                reviews.append(text[:400])
        
        all_text = " ".join(reviews).lower()
        pos_count = sum(all_text.count(w) for w in positive_words)
        neg_count = sum(all_text.count(w) for w in negative_words)
        
        org['положительных_слов'] = pos_count
        org['отрицательных_слов'] = neg_count
        
        
        org['пример_отзыва'] = reviews[0][:200] if reviews else None
        
        print(f"Отзывов: {len(reviews)}")
        
    except Exception as e:
        print(f"Ошибка: {e}")
        org['положительных_слов'] = 0
        org['отрицательных_слов'] = 0
        org['пример_отзыва'] = ""
    
    time.sleep(1.5)

driver.quit()


with open('пансионаты.csv', 'w', newline='', encoding='utf-8-sig') as f:
    fieldnames = ['название', 'адрес', 'рейтинг', 'число отзывов', 'ссылка', 'кол-во по сети', 
                  'цены', 'количество_услуг', 'положительных_слов', 'отрицательных_слов', 'пример_отзыва']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_orgs)



 1: Дружный Берег
Отзывов: 5

 2: Пансионат для пожилых людей Вгармонии
Отзывов: 5

 3: Аннушка
Отзывов: 5

 4: Любимые люди
Отзывов: 5

 5: Kartino Village Club
Отзывов: 5

 6: Пансионат для пожилых Бабушки и Дедушки
Отзывов: 5

 7: SM-pension
Отзывов: 5

 8: Теплые беседы
Отзывов: 5

 9: Климовский дом-интернат для престарелых и инвалидов
Отзывов: 5

 10: Океан милосердия
Отзывов: 5

 11: Оазис
Отзывов: 5

 12: Золотая пора
Отзывов: 5

 13: Академия Долголетия
Отзывов: 5

 14: Теплые беседы
Отзывов: 5

 15: Геронтологический центр Юго-западный филиал Геронтологический центр для ветеранов войны Коньково
Отзывов: 5

 16: Троицк-1
Отзывов: 5

 17: Сениор Групп
Отзывов: 5

 18: Доброта
Отзывов: 5

 19: Дружный Берег
Отзывов: 5

 20: Радость жизни
Отзывов: 5

 21: Опека
Отзывов: 5

 22: Опека
Отзывов: 5

 23: GreenDay
Отзывов: 5

 24: Пансионат Ховрино
Отзывов: 5

 25: Купава Социальный Гериатрический центр
Отзывов: 5

 26: 100+
Отзывов: 5

 27: Vip-пенсионер
Отзывов: 5

 28: Пансионат д